# HairNet Compliance Detection 

### MLflow Training

Each run varies:
- Model size (`yolov26n`, `yolov26m`, `yolo12n`, `yolo12m`)
- Epochs
- Image size
- Augmentation settings

At the end, MLflow lets us compare all runs and pick the winner.

## 0. Install Dependencies

In [ ]:
!pip install ultralytics mlflow python-dotenv -q

## 1. Imports & Setup

In [ ]:
import os
import sys
from pathlib import Path

import mlflow
import mlflow.artifacts
import pandas as pd
from dotenv import load_dotenv
from ultralytics import YOLO

# Load .env 
load_dotenv(Path('.env'))


DATA_YAML = Path('hairnet-compliance-detection-1/data.yaml')
assert DATA_YAML.exists(), f'data.yaml not found at {DATA_YAML}. Run the download cell first.'

# MLflow tracking 

MLFLOW_TRACKING_URI = 'mlruns'
EXPERIMENT_NAME     = 'hairnet-compliance-detection'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'MLflow tracking URI : {mlflow.get_tracking_uri()}')
print(f'Experiment          : {EXPERIMENT_NAME}')
print(f'Data YAML           : {DATA_YAML.resolve()}')

## 2. Define Experiment Configurations

Each dict is one MLflow run. Add or remove entries to control what gets trained.

#### [yolo AUG doc] https://docs.ultralytics.com/guides/yolo-data-augmentation

In [3]:
AUG_OFF = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, translate=0.0,
               scale=0.0, fliplr=0.0, mosaic=0.0, erasing=0.0, auto_augment=None)

AUG_ON = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, translate=0.1, scale=0.5,
              fliplr=0.5, flipud=0.1, mosaic=1.0,
              erasing=0.4, degrees=10.0, auto_augment='randaugment')

RUNS = [
    # ── Baselines (no augmentation) ───────────────────────────────────
    {'run_name': 'yolov26n-baseline', 'model': 'yolov26n.pt', 'epochs': 30, 'batch': 16, **AUG_OFF},
    {'run_name': 'yolov26m-baseline', 'model': 'yolov26m.pt', 'epochs': 30, 'batch': 8,  **AUG_OFF},
    {'run_name': 'yolo12n-baseline',  'model': 'yolo12n.pt',  'epochs': 30, 'batch': 16, **AUG_OFF},
    {'run_name': 'yolo12m-baseline',  'model': 'yolo12m.pt',  'epochs': 30, 'batch': 8,  **AUG_OFF},

    # ── Augmented ─────────────────────────────────────────────────────
    {'run_name': 'yolov26n-augmented', 'model': 'yolov26n.pt', 'epochs': 50, 'batch': 16, **AUG_ON},
    {'run_name': 'yolov26m-augmented', 'model': 'yolov26m.pt', 'epochs': 50, 'batch': 8,  **AUG_ON},
    {'run_name': 'yolo12n-augmented',  'model': 'yolo12n.pt',  'epochs': 50, 'batch': 16, **AUG_ON},
    {'run_name': 'yolo12m-augmented',  'model': 'yolo12m.pt',  'epochs': 50, 'batch': 8,  **AUG_ON},
]

# shared params applied to every run
SHARED = dict(imgsz=640, optimizer='auto', patience=10)

print(f'{len(RUNS)} runs defined:\n')
print(f"  {'Run':<25} {'Model':<14} {'Epochs':>6} {'Batch':>6} {'Aug'}")
print('  ' + '-' * 60)
for r in RUNS:
    aug = 'ON' if r['mosaic'] > 0 else 'OFF'
    print(f"  {r['run_name']:<25} {r['model']:<14} {r['epochs']:>6} {r['batch']:>6}   {aug}")

8 runs defined:

  Run                       Model          Epochs  Batch Aug
  ------------------------------------------------------------
  yolov26n-baseline         yolov26n.pt        30     16   OFF
  yolov26m-baseline         yolov26m.pt        30      8   OFF
  yolo12n-baseline          yolo12n.pt         30     16   OFF
  yolo12m-baseline          yolo12m.pt         30      8   OFF
  yolov26n-augmented        yolov26n.pt        50     16   ON
  yolov26m-augmented        yolov26m.pt        50      8   ON
  yolo12n-augmented         yolo12n.pt         50     16   ON
  yolo12m-augmented         yolo12m.pt         50      8   ON


In [4]:
print(RUNS)

[{'run_name': 'yolov26n-baseline', 'model': 'yolov26n.pt', 'epochs': 30, 'batch': 16, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolov26m-baseline', 'model': 'yolov26m.pt', 'epochs': 30, 'batch': 8, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolo12n-baseline', 'model': 'yolo12n.pt', 'epochs': 30, 'batch': 16, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolo12m-baseline', 'model': 'yolo12m.pt', 'epochs': 30, 'batch': 8, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolov26n-augmented', 'model': 'yolov26n.pt', 'epochs': 50, 'batch': 16, 'hsv_h'

## 3. Training Loop

Each config gets its own MLflow run. Parameters, metrics, and the best weights are all logged automatically.

In [ ]:
FIXED_KEYS = {'run_name', 'model', 'epochs', 'batch'}

for cfg in RUNS:
    run_name = cfg['run_name']
    aug_args = {k: v for k, v in cfg.items() if k not in FIXED_KEYS and v is not None}

    print(f'\n{"="*60}')
    print(f'Starting run: {run_name}')
    print(f'{"="*60}')

    with mlflow.start_run(run_name=run_name):

        # ── Log all parameters 
        mlflow.log_params({
            'model' : cfg['model'],
            'epochs': cfg['epochs'],
            'batch' : cfg['batch'],
            **SHARED,
            **aug_args,
        })

        # ── Train 
        model   = YOLO(cfg['model'])
        results = model.train(
            data     = str(DATA_YAML),
            epochs   = cfg['epochs'],
            batch    = cfg['batch'],
            project  = 'runs/train',
            name     = run_name,
            exist_ok = True,
            verbose  = False,
            **SHARED,
            **aug_args,
        )

        # ── Log final metrics 
        metrics = results.results_dict
        mlflow.log_metrics({
            'mAP50'    : metrics.get('metrics/mAP50(B)',     0),
            'mAP50-95' : metrics.get('metrics/mAP50-95(B)', 0),
            'precision': metrics.get('metrics/precision(B)', 0),
            'recall'   : metrics.get('metrics/recall(B)',    0),
            'box_loss' : metrics.get('val/box_loss',         0),
            'cls_loss' : metrics.get('val/cls_loss',         0),
            'fitness'  : results.fitness
        })
        
        # ── Per-epoch loss curves
        csv_path = Path(f'runs/train/{run_name}/results.csv')
        if csv_path.exists():
            df_results = pd.read_csv(csv_path)
            df_results.columns = df_results.columns.str.strip()
            for _, row in df_results.iterrows():
                epoch = int(row['epoch'])
                mlflow.log_metrics({
                    'train/box_loss': row.get('train/box_loss', 0),
                    'train/cls_loss': row.get('train/cls_loss', 0),
                    'train/dfl_loss': row.get('train/dfl_loss', 0),
                    'val/box_loss'  : row.get('val/box_loss',   0),
                    'val/cls_loss'  : row.get('val/cls_loss',   0),
                    'val/dfl_loss'  : row.get('val/dfl_loss',   0),
                }, step=epoch)

        # ── Log best weights as artifact copy the best to mlflow storage
        best_weights = Path(f'runs/train/{run_name}/weights/best.pt')
        if best_weights.exists():
            mlflow.log_artifact(str(best_weights), artifact_path='weights')

        print(f'Run complete — mAP50: {metrics.get("metrics/mAP50(B)", 0):.4f}')